#  Eye Disease Classification
### Strategy: EfficientNetB5 + EfficientNetV2-M + ConvNeXt-Small Ensemble | 3-Fold CV | CLAHE | MixUp | TTA | Focal Loss | F1 Score

In [ ]:
# CELL 1 — Installs
!pip install -q albumentations==1.3.1
!pip install -q tensorflow_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.7/125.7 kB 6.3 MB/s eta 0:00:00


In [ ]:

# CELL 2 — Imports
import os, cv2, warnings, gc, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, accuracy_score
)

import tensorflow as tf
import tensorflow_hub as hub
from tensorflow import keras
from tensorflow.keras import layers, Model, mixed_precision
from tensorflow.keras.applications import EfficientNetB5, EfficientNetV2M
from tensorflow.keras.callbacks import (
    ModelCheckpoint, EarlyStopping,
    ReduceLROnPlateau, LearningRateScheduler
)
import albumentations as A

warnings.filterwarnings('ignore')

# ── Mixed precision (2x faster on Kaggle T4/P100) ──
mixed_precision.set_global_policy('mixed_float16')

# ── Reproducibility ──
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print('TF  :', tf.__version__)
print('GPU :', tf.config.list_physical_devices('GPU'))

TF  : 2.19.0
GPU : [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [93]:
# ─────────────────────────────────────────────
# CELL 3 — Config
# ─────────────────────────────────────────────
DATA_DIR  = '/kaggle/input/datasets/dhruvdevaliya/eye-diseases/dataset'
SAVE_DIR  = '/kaggle/working/'

IMG_SIZE_B5      = 380   # EfficientNetB5  native size
IMG_SIZE_V2M     = 384   # EfficientNetV2-M native size
IMG_SIZE_CONVNXT = 224   # ConvNeXt-Small  native size

BATCH_SIZE      = 8
N_FOLDS         = 2
EPOCHS_FROZEN   = 6    # phase-1: head only
EPOCHS_FINETUNE = 15   # phase-2: full fine-tune  (EarlyStopping will cut short)
N_CLASSES       = 4

# ConvNeXt-Small — ImageNet-21k pretrained via TF-Hub
CONVNEXT_URL = 'https://tfhub.dev/google/convnext/small/feature_vector/1'

CLASS_NAMES = sorted(os.listdir(DATA_DIR))
print('Classes :', CLASS_NAMES)

Classes : ['cataract', 'diabetic_retinopathy', 'glaucoma', 'normal']


In [94]:
# ─────────────────────────────────────────────
# CELL 4 — Master dataframe
# ─────────────────────────────────────────────
filepaths, labels = [], []
for cls in CLASS_NAMES:
    cls_dir = os.path.join(DATA_DIR, cls)
    for fname in os.listdir(cls_dir):
        filepaths.append(os.path.join(cls_dir, fname))
        labels.append(cls)

df = pd.DataFrame({'filepath': filepaths, 'label': labels})
df['label_idx'] = df['label'].map({c: i for i, c in enumerate(CLASS_NAMES)})
print(df['label'].value_counts())
print('Total images:', len(df))

label
diabetic_retinopathy    1098
normal                  1074
cataract                1038
glaucoma                1007
Name: count, dtype: int64
Total images: 4217


In [95]:
# ─────────────────────────────────────────────
# CELL 5 — CLAHE + Ben Graham preprocessing
# ─────────────────────────────────────────────
def clahe_preprocess(img_bgr):
    lab     = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe   = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l       = clahe.apply(l)
    lab     = cv2.merge([l, a, b])
    img     = cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)
    # Ben Graham: remove global lighting gradient
    img     = cv2.addWeighted(
        img, 4, cv2.GaussianBlur(img, (0, 0), sigmaX=10), -4, 128
    )
    return img

def load_image(path, img_size, apply_clahe=True):
    try:
        img = cv2.imread(path)
        if img is None or img.size == 0:
            img = np.zeros((img_size, img_size, 3), dtype=np.uint8)
        if apply_clahe:
            img = clahe_preprocess(img)
        img = cv2.resize(img, (img_size, img_size))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        return img.astype(np.float32)
    except Exception:
        return np.zeros((img_size, img_size, 3), dtype=np.float32)

In [96]:
# ─────────────────────────────────────────────
# CELL 6 — Albumentations pipelines
# ─────────────────────────────────────────────
train_aug = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Rotate(limit=30, p=0.7),
    A.RandomBrightnessContrast(brightness_limit=0.3,
                               contrast_limit=0.3, p=0.6),
    A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, p=0.4),
    A.GridDistortion(p=0.3),
    A.ElasticTransform(alpha=1, sigma=10, p=0.3),
    A.GaussNoise(var_limit=(5, 30), p=0.3),
    A.CoarseDropout(max_holes=8, max_height=32, max_width=32, p=0.3),
    A.CLAHE(clip_limit=2.0, p=0.3),
    A.Sharpen(p=0.2),
])

val_aug = A.Compose([])

tta_aug = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Rotate(limit=15, p=0.5),
    A.RandomBrightnessContrast(0.1, 0.1, p=0.3),
])

In [97]:
# ─────────────────────────────────────────────
# CELL 7 — MixUp
# ─────────────────────────────────────────────
def mixup(images, labels, alpha=0.3):
    lam          = np.random.beta(alpha, alpha)
    idx          = np.random.permutation(len(images))
    mixed_imgs   = lam * images   + (1 - lam) * images[idx]
    mixed_labels = lam * labels   + (1 - lam) * labels[idx]
    return mixed_imgs, mixed_labels

In [98]:
# ─────────────────────────────────────────────
# CELL 8 — Dataset pipeline (generator-based, no py_function)
# ─────────────────────────────────────────────
class EyeDataGenerator(keras.utils.Sequence):
    def __init__(self, df, img_size, augment_fn,
                 batch_size, shuffle=True, use_mixup=False):
        self.df         = df.reset_index(drop=True)
        self.img_size   = img_size
        self.augment_fn = augment_fn
        self.batch_size = batch_size
        self.shuffle    = shuffle
        self.use_mixup  = use_mixup
        self.indexes    = np.arange(len(self.df))
        if self.shuffle:
            np.random.shuffle(self.indexes)

    def __len__(self):
        return max(1, len(self.df) // self.batch_size)

    def __getitem__(self, idx):
        batch_idx = self.indexes[
            idx * self.batch_size:(idx + 1) * self.batch_size
        ]
        batch     = self.df.iloc[batch_idx]
        imgs, lbls = [], []
        for _, row in batch.iterrows():
            img = load_image(row['filepath'], self.img_size)
            aug = self.augment_fn(image=img.astype(np.uint8))['image']
            imgs.append(aug.astype(np.float32))
            one_hot = np.zeros(N_CLASSES, dtype=np.float32)
            one_hot[int(row['label_idx'])] = 1.0
            lbls.append(one_hot)
        imgs = np.array(imgs, dtype=np.float32)
        lbls = np.array(lbls, dtype=np.float32)
        if self.use_mixup:
            imgs, lbls = mixup(imgs, lbls)
        return imgs, lbls

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indexes)


def build_dataset(df_fold, img_size, augment_fn,
                  batch_size, shuffle=True, use_mixup=False):
    return EyeDataGenerator(
        df=df_fold, img_size=img_size, augment_fn=augment_fn,
        batch_size=batch_size, shuffle=shuffle, use_mixup=use_mixup
    )

In [99]:
# ─────────────────────────────────────────────
# CELL 9 — Focal Loss
# ─────────────────────────────────────────────
def focal_loss(gamma=2.0, alpha=0.25):
    def loss_fn(y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0)
        ce     = -y_true * tf.math.log(y_pred)
        weight = alpha * y_true * tf.pow(1.0 - y_pred, gamma)
        return tf.reduce_mean(tf.reduce_sum(weight * ce, axis=-1))
    return loss_fn

In [100]:
# ─────────────────────────────────────────────
# CELL 10 — Cosine Annealing LR
# ─────────────────────────────────────────────
def cosine_annealing(epoch, lr):
    eta_min = 1e-6
    eta_max = 1e-4
    return eta_min + 0.5 * (eta_max - eta_min) * (
        1 + np.cos(np.pi * (epoch % EPOCHS_FINETUNE) / EPOCHS_FINETUNE)
    )

In [101]:
# ─────────────────────────────────────────────
# CELL 11 — Model builders
#   build_efficientnet  → B5 or V2-M
#   build_convnext      → ConvNeXt-Small via TF-Hub
# ─────────────────────────────────────────────
def build_efficientnet(backbone_fn, img_size, freeze=True):
    inputs = layers.Input(shape=(img_size, img_size, 3))
    x      = layers.Rescaling(1.0 / 255.0)(inputs)
    base   = backbone_fn(
        include_top=False, weights='imagenet', input_tensor=x
    )
    if freeze:
        base.trainable = False
    else:
        base.trainable = True
        for layer in base.layers[:100]:
            if isinstance(layer, layers.BatchNormalization):
                layer.trainable = False

    x = base.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(256, activation='relu',
                     kernel_regularizer=keras.regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(N_CLASSES, activation='softmax',
                           dtype='float32')(x)
    return Model(inputs=base.input, outputs=outputs)


def build_convnext(freeze=True):
    """
    ConvNeXt-Small from TF-Hub (ImageNet-21k pretrained).
    Hub layer is a KerasLayer — trainable flag works normally.
    """
    inputs    = layers.Input(shape=(IMG_SIZE_CONVNXT, IMG_SIZE_CONVNXT, 3))
    # Hub expects [0,1] float input
    x         = layers.Rescaling(1.0 / 255.0)(inputs)
    hub_layer = hub.KerasLayer(
        CONVNEXT_URL,
        trainable=(not freeze),
        arguments=dict(training=(not freeze))
    )
    x       = hub_layer(x)
    x       = layers.BatchNormalization()(x)
    x       = layers.Dropout(0.4)(x)
    x       = layers.Dense(256, activation='relu',
                           kernel_regularizer=keras.regularizers.l2(1e-4))(x)
    x       = layers.Dropout(0.3)(x)
    outputs = layers.Dense(N_CLASSES, activation='softmax',
                           dtype='float32')(x)
    return Model(inputs=inputs, outputs=outputs)


def unfreeze_convnext(model):
    """Unfreeze the hub layer inside a ConvNeXt model."""
    for layer in model.layers:
        if isinstance(layer, hub.KerasLayer):
            layer.trainable = True
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False
    return model

In [102]:
# ─────────────────────────────────────────────
# CELL 12 — TTA inference
# ─────────────────────────────────────────────
def tta_predict(model, df_sub, img_size, n_tta=6):
    all_preds = []
    paths = df_sub['filepath'].values
    for i, path in enumerate(paths):
        img   = load_image(path, img_size)
        preds = []
        for _ in range(n_tta):
            aug = tta_aug(image=img.astype(np.uint8))['image']
            aug = aug.astype(np.float32)[np.newaxis]  # (1, H, W, 3)
            p   = model(aug, training=False).numpy()  # use model() not predict()
            preds.append(p[0])
        all_preds.append(np.mean(preds, axis=0))
        if (i+1) % 50 == 0:
            print(f'    TTA: {i+1}/{len(paths)}')
    return np.array(all_preds)

In [103]:
# ─────────────────────────────────────────────
# CELL 13 — Train one model (phase-1 + phase-2)
# ─────────────────────────────────────────────
def train_model(model_name, build_fn_frozen, build_fn_open,
                img_size, df_tr, df_val, fold):

    ckpt_path = os.path.join(SAVE_DIR, f'best_{model_name}_fold{fold}.keras')

    train_ds = build_dataset(df_tr,  img_size, train_aug,
                             BATCH_SIZE, shuffle=True,  use_mixup=True)
    val_ds   = build_dataset(df_val, img_size, val_aug,
                             BATCH_SIZE, shuffle=False, use_mixup=False)

    # ══════════════════════════════════════
    # PHASE 1 — Frozen backbone, train head
    # ══════════════════════════════════════
    tf.keras.backend.clear_session(); gc.collect()
    model = build_fn_frozen()
    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss=focal_loss(gamma=2.0, alpha=0.25),
        metrics=['accuracy']
    )
    print(f'  [{model_name}] Phase-1: head training...')
    h1 = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS_FROZEN,
        callbacks=[
            EarlyStopping(patience=4, restore_best_weights=True,
                          monitor='val_accuracy', verbose=0),
            ReduceLROnPlateau(factor=0.3, patience=2, min_lr=1e-6,
                              monitor='val_accuracy', verbose=0),
        ],
        verbose=1
    )

    # ══════════════════════════════════════
    # PHASE 2 — Unfreeze full backbone
    # ══════════════════════════════════════
    if build_fn_open is not None:        # ConvNeXt path
        weights = model.get_weights()
        del model; gc.collect()
        tf.keras.backend.clear_session()
        model = build_fn_open()
        model.set_weights(weights)
    else:                                # EfficientNet path — unfreeze in-place
        for layer in model.layers:
            layer.trainable = True
            if isinstance(layer, layers.BatchNormalization):
                layer.trainable = False

    model.compile(
        optimizer=keras.optimizers.Adam(1e-4),
        loss=focal_loss(gamma=2.0, alpha=0.25),
        metrics=['accuracy']
    )
    print(f'  [{model_name}] Phase-2: full fine-tune...')
    h2 = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS_FINETUNE,
        callbacks=[
            ModelCheckpoint(ckpt_path, save_best_only=True,
                            monitor='val_accuracy', verbose=0),
            EarlyStopping(patience=6, restore_best_weights=True,
                          monitor='val_accuracy', verbose=1),
            LearningRateScheduler(cosine_annealing, verbose=0),
        ],
        verbose=1
    )
    return model, h1, h2

In [ ]:
# ─────────────────────────────────────────────
# CELL 14 — 3-Fold CV training loop WITH RESUME
# ─────────────────────────────────────────────

# ── Hold-out test split (10 %) ──
df_trainval, df_test = train_test_split(
    df, test_size=0.10, stratify=df['label'], random_state=SEED
)
df_trainval = df_trainval.reset_index(drop=True)
df_test     = df_test.reset_index(drop=True)
print(f'Train+Val: {len(df_trainval)}  |  Test: {len(df_test)}')

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

test_preds_all = []
fold_histories = []

for fold, (tr_idx, val_idx) in enumerate(
        skf.split(df_trainval, df_trainval['label'])):

    print(f'\n{"="*60}')
    print(f'  FOLD {fold+1} / {N_FOLDS}')
    print(f'{"="*60}')

    df_tr  = df_trainval.iloc[tr_idx].reset_index(drop=True)
    df_val = df_trainval.iloc[val_idx].reset_index(drop=True)

    # ── EfficientNetB5 ───────────────────────
    b5_ckpt = os.path.join(SAVE_DIR, f'best_EfficientNetB5_fold{fold+1}.keras')
    if os.path.exists(b5_ckpt):
        print(f'  [EfficientNetB5] ✅ Checkpoint found — loading & skipping training')
        model_b5 = keras.models.load_model(b5_ckpt, custom_objects={'loss_fn': focal_loss()})
        h1_b5 = type('H', (), {'history': {'accuracy': [0], 'val_accuracy': [0], 'loss': [0], 'val_loss': [0]}})()
        h2_b5 = type('H', (), {'history': {'accuracy': [0], 'val_accuracy': [0], 'loss': [0], 'val_loss': [0]}})()
    else:
        b5_size = IMG_SIZE_B5
        model_b5, h1_b5, h2_b5 = train_model(
            model_name      = 'EfficientNetB5',
            build_fn_frozen = lambda: build_efficientnet(EfficientNetB5, b5_size, freeze=True),
            build_fn_open   = None,
            img_size        = IMG_SIZE_B5,
            df_tr=df_tr, df_val=df_val, fold=fold+1
        )
    tp_b5 = tta_predict(model_b5, df_test, IMG_SIZE_B5)
    test_preds_all.append(tp_b5)
    fold_histories.append(('EfficientNetB5', fold+1, h1_b5, h2_b5))
    del model_b5; gc.collect(); tf.keras.backend.clear_session()

    # ── EfficientNetV2-M ─────────────────────
    v2m_ckpt = os.path.join(SAVE_DIR, f'best_EfficientNetV2M_fold{fold+1}.keras')
    if os.path.exists(v2m_ckpt):
        print(f'  [EfficientNetV2M] ✅ Checkpoint found — loading & skipping training')
        model_v2m = keras.models.load_model(v2m_ckpt, custom_objects={'loss_fn': focal_loss()})
        h1_v2m = type('H', (), {'history': {'accuracy': [0], 'val_accuracy': [0], 'loss': [0], 'val_loss': [0]}})()
        h2_v2m = type('H', (), {'history': {'accuracy': [0], 'val_accuracy': [0], 'loss': [0], 'val_loss': [0]}})()
    else:
        v2m_size = IMG_SIZE_V2M
        model_v2m, h1_v2m, h2_v2m = train_model(
            model_name      = 'EfficientNetV2M',
            build_fn_frozen = lambda: build_efficientnet(EfficientNetV2M, v2m_size, freeze=True),
            build_fn_open   = None,
            img_size        = IMG_SIZE_V2M,
            df_tr=df_tr, df_val=df_val, fold=fold+1
        )
    tp_v2m = tta_predict(model_v2m, df_test, IMG_SIZE_V2M)
    test_preds_all.append(tp_v2m)
    fold_histories.append(('EfficientNetV2M', fold+1, h1_v2m, h2_v2m))
    del model_v2m; gc.collect(); tf.keras.backend.clear_session()

    # ── ConvNeXt-Small (fold 1 only) ──
    if fold == 0:
        cnx_ckpt = os.path.join(SAVE_DIR, f'best_ConvNeXt_fold{fold+1}.keras')
        if os.path.exists(cnx_ckpt):
            print(f'  [ConvNeXt] ✅ Checkpoint found — loading & skipping training')
            model_cnx = keras.models.load_model(cnx_ckpt, custom_objects={'loss_fn': focal_loss()})
            h1_cnx = type('H', (), {'history': {'accuracy': [0], 'val_accuracy': [0], 'loss': [0], 'val_loss': [0]}})()
            h2_cnx = type('H', (), {'history': {'accuracy': [0], 'val_accuracy': [0], 'loss': [0], 'val_loss': [0]}})()
        else:
            print('\n  [ConvNeXt-Small] Fold-1 only (time budget)')
            model_cnx, h1_cnx, h2_cnx = train_model(
                model_name      = 'ConvNeXt',
                build_fn_frozen = lambda: build_convnext(freeze=True),
                build_fn_open   = lambda: build_convnext(freeze=False),
                img_size        = IMG_SIZE_CONVNXT,
                df_tr=df_tr, df_val=df_val, fold=fold+1
            )
        tp_cnx = tta_predict(model_cnx, df_test, IMG_SIZE_CONVNXT)
        test_preds_all.append(tp_cnx)
        fold_histories.append(('ConvNeXt', fold+1, h1_cnx, h2_cnx))
        del model_cnx; gc.collect(); tf.keras.backend.clear_session()

print('\n✅ All training complete!')

Train+Val: 3795  |  Test: 422

  FOLD 1 / 2
  [EfficientNetB5] Phase-1: head training...
Epoch 1/6
237/237 ━━━━━━━━━━━━━━━━━━━━ 225s 718ms/step - accuracy: 0.3341 - loss: 0.6060 - val_accuracy: 0.4673 - val_loss: 0.1967 - learning_rate: 0.0010
Epoch 2/6
237/237 ━━━━━━━━━━━━━━━━━━━━ 154s 652ms/step - accuracy: 0.3699 - loss: 0.4919 - val_accuracy: 0.5222 - val_loss: 0.2097 - learning_rate: 0.0010
Epoch 3/6
237/237 ━━━━━━━━━━━━━━━━━━━━ 154s 650ms/step - accuracy: 0.3776 - loss: 0.3424 - val_accuracy: 0.4726 - val_loss: 0.3503 - learning_rate: 0.0010
Epoch 4/6
237/237 ━━━━━━━━━━━━━━━━━━━━ 153s 647ms/step - accuracy: 0.4025 - loss: 0.2682 - val_accuracy: 0.6988 - val_loss: 0.1350 - learning_rate: 0.0010
Epoch 5/6
237/237 ━━━━━━━━━━━━━━━━━━━━ 151s 640ms/step - accuracy: 0.3973 - loss: 0.2103 - val_accuracy: 0.6334 - val_loss: 0.1663 - learning_rate: 0.0010
Epoch 6/6
237/237 ━━━━━━━━━━━━━━━━━━━━ 154s 649ms/step - accuracy: 0.4497 - loss: 0.1985 - val_accuracy: 0.6308 - val_loss: 0.1520 - lea

2026-04-29 07:07:44.993199: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-29 07:07:45.184008: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-29 07:07:45.778740: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-29 07:07:45.969933: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-29 07:07:46.430495: E external/local_xla/xla/stream_

237/237 ━━━━━━━━━━━━━━━━━━━━ 304s 782ms/step - accuracy: 0.3486 - loss: 0.2636 - val_accuracy: 0.6086 - val_loss: 0.1546 - learning_rate: 1.0000e-04
Epoch 2/15
237/237 ━━━━━━━━━━━━━━━━━━━━ 159s 669ms/step - accuracy: 0.5818 - loss: 0.1555 - val_accuracy: 0.5754 - val_loss: 0.1464 - learning_rate: 9.8918e-05
Epoch 3/15
237/237 ━━━━━━━━━━━━━━━━━━━━ 0s 450ms/step - accuracy: 0.6505 - loss: 0.1369

In [ ]:
# ─────────────────────────────────────────────
# CELL 15 — Final ensemble + metrics
# ─────────────────────────────────────────────
final_preds   = np.mean(test_preds_all, axis=0)   # (n_test, 4)
final_classes = np.argmax(final_preds, axis=1)
true_classes  = df_test['label_idx'].values

acc    = accuracy_score(true_classes, final_classes)
f1_mac = f1_score(true_classes, final_classes, average='macro')
f1_wt  = f1_score(true_classes, final_classes, average='weighted')
f1_per = f1_score(true_classes, final_classes, average=None)

print('=' * 55)
print(f'   Ensemble Test Accuracy      : {acc:.4f}')
print(f'   Macro   F1-Score            : {f1_mac:.4f}')
print(f'   Weighted F1-Score           : {f1_wt:.4f}')
print('  Per-class F1:')
for cls, f1 in zip(CLASS_NAMES, f1_per):
    print(f'    {cls:<20}: {f1:.4f}')
print('=' * 55)

print('\nFull Classification Report:')
print(classification_report(
    true_classes, final_classes, target_names=CLASS_NAMES
))

In [ ]:
# ─────────────────────────────────────────────
# CELL 16 — Confusion Matrix
# ─────────────────────────────────────────────
cm = confusion_matrix(true_classes, final_classes)
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title(f'Confusion Matrix — Ensemble (TTA)\nAcc={acc:.4f}  MacroF1={f1_mac:.4f}')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'confusion_matrix.png'), dpi=150)
plt.show()

In [ ]:
# ─────────────────────────────────────────────
# CELL 17 — Per-class F1 bar chart
# ─────────────────────────────────────────────
plt.figure(figsize=(8, 4))
bars = plt.bar(CLASS_NAMES, f1_per,
               color=['#4C72B0','#DD8452','#55A868','#C44E52'])
plt.axhline(f1_mac, color='black', linestyle='--',
            label=f'Macro F1 = {f1_mac:.4f}')
for bar, val in zip(bars, f1_per):
    plt.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.005,
             f'{val:.3f}', ha='center', va='bottom', fontsize=11)
plt.ylim(0, 1.05)
plt.title('Per-class F1 Score — Ensemble')
plt.ylabel('F1 Score')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'f1_per_class.png'), dpi=150)
plt.show()

In [ ]:
# ─────────────────────────────────────────────
# CELL 18 — Training curves
# ─────────────────────────────────────────────
n_plots = len(fold_histories)
fig, axes = plt.subplots(n_plots, 2, figsize=(14, 5 * n_plots))
if n_plots == 1:
    axes = [axes]   # keep indexing consistent

for i, (model_name, fold, h1, h2) in enumerate(fold_histories):
    acc_tr   = h1.history['accuracy']     + h2.history['accuracy']
    acc_val  = h1.history['val_accuracy'] + h2.history['val_accuracy']
    loss_tr  = h1.history['loss']         + h2.history['loss']
    loss_val = h1.history['val_loss']     + h2.history['val_loss']
    split_ep = len(h1.history['accuracy']) - 1

    ax = axes[i]
    ax[0].plot(acc_tr,  label='Train')
    ax[0].plot(acc_val, label='Val')
    ax[0].axvline(split_ep, color='gray', linestyle='--', label='Unfreeze')
    ax[0].set_title(f'{model_name} Fold {fold} — Accuracy')
    ax[0].legend()

    ax[1].plot(loss_tr,  label='Train')
    ax[1].plot(loss_val, label='Val')
    ax[1].axvline(split_ep, color='gray', linestyle='--', label='Unfreeze')
    ax[1].set_title(f'{model_name} Fold {fold} — Loss')
    ax[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'training_curves.png'), dpi=150)
plt.show()

In [ ]:
# ─────────────────────────────────────────────
# CELL 19 — Save predictions CSV
# ─────────────────────────────────────────────
results_df = df_test.copy()
results_df['predicted'] = [CLASS_NAMES[i] for i in final_classes]
results_df['correct']   = results_df['label'] == results_df['predicted']
for i, cls in enumerate(CLASS_NAMES):
    results_df[f'prob_{cls}'] = final_preds[:, i]

results_df.to_csv(
    os.path.join(SAVE_DIR, 'test_predictions.csv'), index=False
)

print('Saved → /kaggle/working/test_predictions.csv')
print(f'\nFinal Accuracy  : {acc:.4f}')
print(f'Macro F1        : {f1_mac:.4f}')
print(f'Weighted F1     : {f1_wt:.4f}')

In [ ]:
# ─────────────────────────────────────────────
# CELL 14b — Load & Save best models after training
# ─────────────────────────────────────────────
import glob

# Find all saved checkpoints
ckpt_files = glob.glob(os.path.join(SAVE_DIR, 'best_*.keras'))
print('Found checkpoints:')
for f in ckpt_files:
    print(' ', f)

# Load each and save as final
loaded_models = {}
for ckpt in ckpt_files:
    name = os.path.basename(ckpt).replace('.keras', '')
    print(f'Loading {name}...')
    m = keras.models.load_model(
        ckpt,
        custom_objects={'loss_fn': focal_loss()}
    )
    # Save in SavedModel format (most compatible)
    save_path = os.path.join(SAVE_DIR, f'FINAL_{name}.keras')
    m.save(save_path)
    print(f'  Saved → {save_path}')
    loaded_models[name] = m

print(f'\n✅ {len(loaded_models)} models saved!')
print('Files in /kaggle/working/:')
for f in sorted(os.listdir(SAVE_DIR)):
    if f.endswith('.keras'):
        size_mb = os.path.getsize(os.path.join(SAVE_DIR, f)) / 1e6
        print(f'  {f}  ({size_mb:.1f} MB)')

In [ ]:
# runn this after cell 14 to save 